# Module 5: LLM-as-judge that you can trust

1. Our judge for "Assumes device", built from the rubric.
2. **Exercise 8 (25 min):** build your own judge, align it on the dev set, run it once on the test set.
3. Correcting a pass rate for an imperfect judge.
4. **Exercise 9 (demo):** the swap test for position bias.

Judges need an API key. Without one, you'll see a recorded run if the facilitator has saved one.

In [ ]:
# Setup: run this cell first. It works in Google Colab and on your own laptop.
import os, sys
REPO_URL = "https://github.com/MarinaWyss/evaluating-ai-systems"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/EvalsWorkshop"):
        !git clone -q {REPO_URL} /content/EvalsWorkshop
        !pip install -q "litellm>=1.80.5" tenacity
    os.chdir("/content/EvalsWorkshop")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import pandas as pd
from beefcake import llm
llm.load_colab_secrets()
if llm.has_api_key():
    print("API key found. Bot model:", llm.get_model())
else:
    print("No API key found, so this notebook runs in offline mode with pre-generated data.")

## 1. A real judge

Overview, one failure mode, PASS and FAIL criteria, reasoning before the verdict, examples, then the trace.

In [ ]:
from beefcake.judge import ASSUMES_DEVICE_JUDGE, build_judge_prompt, judge_all, run_judge, tpr_tnr
print(ASSUMES_DEVICE_JUDGE)

## 2. Exercise 8: Judge alignment (25 min)

1. Build an **Assumes device** judge with the six-part template (8 min). Start from your Exercise 4a rubric if
   you wrote it for Assumes device, or from the worked example in `rubric/rubric_template.md`.
2. Run it on the **dev** set. Report TPR and TNR (5 min).
3. Read the reasoning on every disagreement. Improve the prompt once. Rerun (8 min).
4. **One** run on the **test** set. Post your numbers (4 min).

Every group builds the same judge, because these traces are labeled for Assumes device only. They're split
about 10% train (for your few-shot examples), 40% dev, and 50% test.

In [ ]:
traces = pd.read_csv("data/exercise8_labeled_traces.csv")
train, dev, test = (traces[traces["Split"] == s] for s in ["train", "dev", "test"])
print(len(train), "train,", len(dev), "dev,", len(test), "test")
display(traces.groupby("Split")["Assumes device"].value_counts().unstack())
train[["User Query", "AI Response", "Assumes device", "Notes"]]

In [ ]:
# Build your judge from your rubric. Examples come from the TRAIN split only.
examples = [
    {"user_query": r["User Query"], "ai_response": r["AI Response"], "judgment": r["Assumes device"], "reasoning": r["Notes"]}
    for _, r in pd.concat([train[(train["Assumes device"] == "PASS") & ~train["Notes"].str.contains("different failure")].head(2),
                           train[train["Assumes device"] == "FAIL"].head(2)]).iterrows()
]  # pick your own: the most useful examples are the borderline ones

my_judge = build_judge_prompt(
    failure_mode="Assumes device",
    definition="YOUR DEFINITION FROM THE RUBRIC",
    pass_criteria="YOUR PASS CRITERIA",
    fail_criteria="YOUR FAIL CRITERIA",
    examples=examples,
)
print(my_judge)

### Run it on the dev set

In [ ]:
from pathlib import Path

RECORDED = Path("data/example_runs/judge_assumes_device.csv")

if llm.has_api_key():
    dev_run = judge_all(dev, my_judge)
elif RECORDED.exists():
    print("No API key: showing the recorded run of OUR judge, not yours.")
    dev_run = pd.read_csv(RECORDED)
    dev_run = dev_run[dev_run["Split"] == "dev"]
else:
    dev_run = None
    print("Needs an API key (or a recorded run in data/example_runs/).")

if dev_run is not None:
    print(tpr_tnr(dev_run["Assumes device"], dev_run["Judge"]))

### Read every disagreement

Is the prompt ambiguous? Is an edge case missing? Or was the human label wrong?

In [ ]:
if dev_run is not None:
    wrong = dev_run[dev_run["Judge"] != dev_run["Assumes device"]]
    for _, r in wrong.iterrows():
        print(f"{r['Trace ID']}  human={r['Assumes device']}  judge={r['Judge']}")
        print(f"  USER:  {r['User Query']}\n  BOT:   {r['AI Response']}\n  JUDGE: {r['Judge reasoning']}\n")

Improve the prompt once and rerun the dev cell. When you're happy, run the test set **once**. Your leaderboard
score is the lower of TPR and TNR, so a judge that says PASS to everything can't win.

In [ ]:
RUN_TEST = False  # set to True once, when you're done iterating on dev

if RUN_TEST and llm.has_api_key():
    test_run = judge_all(test, my_judge)
    rates = tpr_tnr(test_run["Assumes device"], test_run["Judge"])
    print(rates, "  leaderboard score:", min(rates["TPR"], rates["TNR"]))

## 3. Correcting for an imperfect judge

If your judge has TPR 89% and TNR 87% and says 80% of production traces pass, the true pass rate is about 88%:
`(observed + TNR - 1) / (TPR + TNR - 1)`. It only works if TPR + TNR is above 1.

In [ ]:
from beefcake.judge import corrected_pass_rate, rogan_gladen

print(f"Worked example: {rogan_gladen(0.80, 0.89, 0.87):.1%}")

With a labeled test set, you can also get an interval. The production verdicts below come from a synthetic
production log (made up for this demo). Notice how wide the interval is with a small test set.

In [ ]:
production = pd.read_csv("data/production_log_synthetic.csv", keep_default_na=False)
judged = production.loc[production["Judge: Assumes device"] != "", "Judge: Assumes device"]
if "test_run" in globals():
    labeled_run = test_run                     # your judge's one test-set run
elif RECORDED.exists():
    labeled_run = pd.read_csv(RECORDED)
    labeled_run = labeled_run[labeled_run["Split"] == "test"]   # the recorded run of our judge
else:
    labeled_run = None
    print("Needs a judge run on the test set: yours (RUN_TEST above) or a recorded one.")
if labeled_run is not None:
    print(corrected_pass_rate(labeled_run["Assumes device"], labeled_run["Judge"], judged))

## 4. Exercise 9: The swap test (demo)

A pairwise judge sees two replies and picks the better one. Run every pair in both orders: a consistent judge
picks the same reply both times. In each pair here, both replies give the same answer and one adds friendly
filler, and the judge has to answer "1" or "2", with no option for a tie.

In [ ]:
from beefcake.judge import swap_test

pairs = pd.read_csv("data/exercise9_pairs.csv")
SWAPS = Path("data/example_runs/swap_test.csv")
if llm.has_api_key():
    swaps = swap_test(pairs)
elif SWAPS.exists():
    swaps = pd.read_csv(SWAPS)
    print("Recorded run with", swaps["Judge model"].iloc[0])
else:
    swaps = None
    print("Needs an API key (or a recorded run in data/example_runs/).")
if swaps is not None:
    print(f"{(~swaps['Consistent'].astype(bool)).sum()} of {len(swaps)} verdicts flipped when the order changed")
    display(swaps)